In [1]:
import os

import pandas as pd
import numpy as np
import pickle

#NETTOYAGE
import re
import string
import nltk
from nltk.corpus import stopwords
from gensim.utils import simple_preprocess
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

from sklearn.model_selection import train_test_split

## CONSTANTES

In [2]:
MAX_WORDS = 10000
MAX_LEN = 100
EMBEDDING_DIM = 64
EPOCHS = 3
BATCH_SIZE = 32

MODEL_NAME = "modele_lstm"
TOKEN_NAME = "tokenizer.pickle"
OUTPUT_DIR = "../LSTM"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

## Importation

In [3]:
columns_name = ['TARGET', 'id', 'date', '??', 'user', 'tweet']

df = pd.read_csv("../../../Data/training.1600000.processed.noemoticon.csv", encoding='ISO-8859-1', names=columns_name)

## Nettoyage

In [4]:
#Initialisation
stop_words = set(stopwords.words('english'))

for neg in ['not', 'no', 'never', "n't", 'nor']:
    if neg in stop_words:
        stop_words.remove(neg)
lemmatizer = WordNetLemmatizer()

In [5]:
#Fonction
def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return ""

    #Passer en minuscule tout le texte
    texte = texte.lower()

    #Supprimer des éléments frequents dans des tweets, mais jugés ininteressants pour l'analyse
    texte = re.sub(r'<.*?>', '', texte)
    texte = re.sub(r'@\w+', '', texte)
    texte = re.sub(r'\d+', '', texte)
    texte = re.sub(r'https?://\S+|www\.\S+', '', texte)

    #Supprimer les ponctuactions
    texte = texte.translate(str.maketrans("","",string.punctuation))

    texte = re.sub(r'\s+', ' ', texte)


    #TOKENISATION
    tokens = texte.split()

    #Parcours des tokens:
    clean_tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words and len(token) > 2
    ]

    #Renvoie des elements à joindre
    return " ".join(clean_tokens)


In [6]:
#Application
df_trt = df.copy()[['TARGET', 'id', 'date','user', 'tweet']]

df_trt['tweet_net'] = df_trt['tweet'].apply(nettoyer_texte)

## Tokenisation

In [7]:
tokenizer = Tokenizer(num_words=MAX_WORDS)
# IMPORTANT : On n'apprend le vocabulaire QUE sur le train set pour éviter la fuite de données
tokenizer.fit_on_texts(df_trt['tweet_net'])
sequences = tokenizer.texts_to_sequences(df_trt['tweet_net'])
padded_sequences = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
df_trt['TARGET'] = df_trt['TARGET'].replace(4, 1)

X_train, X_test, y_train, y_test = train_test_split(padded_sequences, df_trt['TARGET'], test_size=0.2, random_state=42)

## LSTM

### Construction du modèle

In [8]:
model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    
    # Bidirectional permet au LSTM de lire dans les deux sens (meilleur contexte)
    Bidirectional(LSTM(64, return_sequences=False)),
    
    Dropout(0.5), # Pour éviter le surapprentissage
    
    Dense(32, activation='relu'),
    Dropout(0.5),
    
    Dense(1, activation='sigmoid') # Sortie binaire (0 ou 1)
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

F:\Applications\Anaconda\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Entrainement du modèle

In [9]:
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test),
    verbose=1
)

Epoch 1/3
15345/40000 ━━━━━━━━━━━━━━━━━━━━ 12:15 30ms/step - accuracy: 0.7515 - loss: 0.5101

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



40000/40000 ━━━━━━━━━━━━━━━━━━━━ 1344s 34ms/step - accuracy: 0.7855 - loss: 0.4641 - val_accuracy: 0.7954 - val_loss: 0.4352
Epoch 2/3
40000/40000 ━━━━━━━━━━━━━━━━━━━━ 1389s 35ms/step - accuracy: 0.8012 - loss: 0.4371 - val_accuracy: 0.7995 - val_loss: 0.4308
Epoch 3/3
40000/40000 ━━━━━━━━━━━━━━━━━━━━ 1173s 29ms/step - accuracy: 0.8071 - loss: 0.4266 - val_accuracy: 0.7999 - val_loss: 0.4316


### Sauvegarde du modèle

In [10]:
#TOKEN
tokenizer_path = os.path.join(OUTPUT_DIR, TOKEN_NAME)
with open(tokenizer_path, 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

#MODELE
model_path = os.path.join(OUTPUT_DIR, MODEL_NAME)
model.export(model_path)

"""print("\n💾 Sauvegarde...")
save_path = os.path.join(OUTPUT_DIR, MODEL_NAME)

# SOLUTION RECOMMANDÉE POUR BERT : save_pretrained
# Cela sauvegarde le modèle ET le tokenizer au bon format automatiquement
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Modèle BERT sauvegardé dans : {save_path}")
print("Tu trouveras dedans 'config.json', 'tf_model.h5' (ou .safetensors) et les fichiers tokenizer.")"""

INFO:tensorflow:Assets written to: ../LSTM\modele_lstm\assets


INFO:tensorflow:Assets written to: ../LSTM\modele_lstm\assets


Saved artifact at '../LSTM\modele_lstm'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 100), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2735914272336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2735914272144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2735914273296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2735914273872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2735914274064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2735914274448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2735914275408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2735914272528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2735914272720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2735914276176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2735914276368: TensorSpec(shape=(), dty

'print("\n💾 Sauvegarde...")\nsave_path = os.path.join(OUTPUT_DIR, MODEL_NAME)\n\n# SOLUTION RECOMMANDÉE POUR BERT : save_pretrained\n# Cela sauvegarde le modèle ET le tokenizer au bon format automatiquement\nmodel.save_pretrained(save_path)\ntokenizer.save_pretrained(save_path)\n\nprint(f"✅ Modèle BERT sauvegardé dans : {save_path}")\nprint("Tu trouveras dedans \'config.json\', \'tf_model.h5\' (ou .safetensors) et les fichiers tokenizer.")'

In [11]:
nouveau_texte = "I LOVE @Health4UandPets u guys r the best!!"
# Ou un exemple négatif :
# nouveau_texte = "Late arrival and rude staff."

# 2. PRÉTRAITEMENT (Crucial : doit être identique au train)
# a. Convertir en séquence d'entiers via le tokenizer entraîné
# ⚠️ Attention : on met des crochets [] car la fonction attend une liste de textes
sequence = tokenizer.texts_to_sequences([nouveau_texte])

# b. Appliquer le Padding (pour avoir la longueur MAX_LEN, ex: 100 ou 128)
# Utilise bien le même MAX_LEN que pour X_train !
data_input = pad_sequences(sequence, maxlen=MAX_LEN, padding='post', truncating='post')

# 3. PRÉDICTION
# Le modèle attend un batch, mais ici on lui donne un batch de taille 1
prediction_brute = model.predict(data_input)

# 4. INTERPRÉTATION
# Le résultat est un tableau (ex: [[0.89]]). On récupère la valeur.
score = prediction_brute[0][0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step


In [12]:
print(score)

0.98213136
